In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [3]:
df = pd.read_csv('../Downloads/415files/dataset1.csv')
df.columns = df.columns.str.strip()
df = df.drop('index', axis=1)
#xgboost requires labels in {0,1}
y = (df['Result'] == 1).astype(int)
X = df.drop('Result', axis=1)

In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

In [5]:
xgb_grid = GridSearchCV(XGBClassifier(random_state=42, eval_metric='logloss'), {'n_estimators': [100, 200], 'max_depth': [3, 5], 'learning_rate': [0.1]}, cv=3, n_jobs=-1)
xgb_grid.fit(X_train_s, y_train)
bp = xgb_grid.best_params_
print(bp)

{'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}


In [6]:
poly = PolynomialFeatures(degree=2, interaction_only=True, include_bias=False)
X_train_fe = poly.fit_transform(X_train_s)
X_test_fe = poly.transform(X_test_s)
xgb_fe = XGBClassifier(random_state=42, eval_metric='logloss', **bp).fit(X_train_fe, y_train)
pred_fe = xgb_fe.predict(X_test_fe)
print(accuracy_score(y_test, pred_fe))
print(confusion_matrix(y_test, pred_fe, labels=[0, 1]))
print(classification_report(y_test, pred_fe, digits=4))

0.973767526006332
[[ 940   40]
 [  18 1213]]
              precision    recall  f1-score   support

           0     0.9812    0.9592    0.9701       980
           1     0.9681    0.9854    0.9767      1231

    accuracy                         0.9738      2211
   macro avg     0.9746    0.9723    0.9734      2211
weighted avg     0.9739    0.9738    0.9737      2211



In [7]:
pca = PCA(n_components=30, random_state=42)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca = pca.transform(X_test_s)
xgb_pca = XGBClassifier(random_state=42, eval_metric='logloss', **bp).fit(X_train_pca, y_train)
pred_pca = xgb_pca.predict(X_test_pca)
print(accuracy_score(y_test, pred_pca))
print(confusion_matrix(y_test, pred_pca, labels=[0, 1]))
print(classification_report(y_test, pred_pca, digits=4))

0.9696969696969697
[[ 933   47]
 [  20 1211]]
              precision    recall  f1-score   support

           0     0.9790    0.9520    0.9653       980
           1     0.9626    0.9838    0.9731      1231

    accuracy                         0.9697      2211
   macro avg     0.9708    0.9679    0.9692      2211
weighted avg     0.9699    0.9697    0.9696      2211

